In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

OUTDIR = Path("synthetic30_tree_distances")
OUTDIR.mkdir(exist_ok=True)

N = 30
SEED = 20260606

# Scale distances to look roughly like mtDNA p-distances.
MAX_DISTANCE = 0.25

MISSING_LEVELS = [0.30, 0.50, 0.65, 0.85]
N_REPLICATES = 30

T_LABELS = [f"T{i+1}" for i in range(N)]
FULL_LABELS = [f"Synthetic_taxon_{i+1:02d}" for i in range(N)]


# ============================================================
# SYNTHETIC TREE DISTANCE MATRICES
# ============================================================

def generate_ultrametric_distance_matrix(n=30, seed=1, max_distance=0.25):
    """
    Generate an exact ultrametric distance matrix from a random rooted binary tree.

    Construction:
    - Start with n singleton clusters.
    - Randomly merge two clusters at increasing heights.
    - Distance between leaves from two newly merged clusters is 2 * merge_height.
    - Rescale the final matrix so that max pairwise distance equals max_distance.

    The resulting matrix satisfies the ultrametric condition:
        d_ij <= max(d_ik, d_jk),
    and for every triple the two largest distances are equal.
    """
    rng = np.random.default_rng(seed)

    clusters = [[i] for i in range(n)]
    D = np.zeros((n, n), dtype=float)

    current_height = 0.0

    while len(clusters) > 1:
        a, b = rng.choice(len(clusters), size=2, replace=False)
        a, b = sorted([a, b])

        cluster_b = clusters.pop(b)
        cluster_a = clusters.pop(a)

        current_height += rng.exponential(scale=1.0)

        for i in cluster_a:
            for j in cluster_b:
                D[i, j] = D[j, i] = 2.0 * current_height

        clusters.append(cluster_a + cluster_b)

    off = D[np.triu_indices(n, k=1)]
    D *= max_distance / off.max()

    np.fill_diagonal(D, 0.0)
    return D


def generate_additive_non_ultrametric_matrix(
    D_ultra,
    seed=1,
    pendant_scale=0.03,
    max_distance=0.25,
):
    """
    Generate an additive but non-ultrametric tree-like matrix.

    Starting from an ultrametric distance matrix, we add leaf-specific pendant
    lengths p_i + p_j. This preserves a tree-like additive structure but
    generally breaks exact ultrametricity.

    This matrix is useful as a robustness / heterogeneity stress test.
    """
    rng = np.random.default_rng(seed)
    n = D_ultra.shape[0]

    pendant = rng.exponential(scale=pendant_scale, size=n)

    D = D_ultra.copy()

    for i in range(n):
        for j in range(i + 1, n):
            D[i, j] += pendant[i] + pendant[j]
            D[j, i] = D[i, j]

    off = D[np.triu_indices(n, k=1)]
    D *= max_distance / off.max()

    np.fill_diagonal(D, 0.0)
    return D


# ============================================================
# MASKING
# ============================================================

def make_missing_mask(n, missing_fraction, seed):
    """
    Create a symmetric boolean missing mask for off-diagonal entries.

    True  = missing entry
    False = observed entry

    Diagonal is always False.
    """
    rng = np.random.default_rng(seed)

    iu = np.triu_indices(n, k=1)
    n_pairs = len(iu[0])
    n_missing = int(round(missing_fraction * n_pairs))

    selected = rng.choice(n_pairs, size=n_missing, replace=False)

    mask = np.zeros((n, n), dtype=bool)

    rows = iu[0][selected]
    cols = iu[1][selected]

    mask[rows, cols] = True
    mask[cols, rows] = True

    np.fill_diagonal(mask, False)
    return mask


def apply_missing_mask(D, mask):
    """
    Return matrix with missing entries replaced by NaN.
    """
    D_obs = D.copy()
    D_obs[mask] = np.nan
    np.fill_diagonal(D_obs, 0.0)
    return D_obs


# ============================================================
# DIAGNOSTICS
# ============================================================

def max_ultrametric_violation(D):
    """
    For each triple, sort the three distances.
    In an ultrametric matrix, the two largest distances must be equal.

    Returns:
        max |largest - second_largest|.
    """
    n = D.shape[0]
    max_v = 0.0

    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                vals = sorted([D[i, j], D[i, k], D[j, k]])
                violation = abs(vals[2] - vals[1])
                max_v = max(max_v, violation)

    return max_v


def matrix_summary(name, D):
    off = D[np.triu_indices_from(D, k=1)]

    print(f"\n{name}")
    print(f"  shape:  {D.shape}")
    print(f"  min:    {off.min():.6f}")
    print(f"  median: {np.median(off):.6f}")
    print(f"  mean:   {off.mean():.6f}")
    print(f"  max:    {off.max():.6f}")
    print(f"  nan:    {np.isnan(off).sum()}")
    print(f"  max ultrametric violation: {max_ultrametric_violation(D):.12e}")


# ============================================================
# SAVE HELPERS
# ============================================================

def save_matrix(M, labels, path):
    df = pd.DataFrame(M, index=labels, columns=labels)
    df.to_csv(path)
    print(f"Saved: {path}")


def save_label_mapping(path):
    pd.DataFrame({
        "T_label": T_LABELS,
        "full_label": FULL_LABELS,
    }).to_csv(path, index=False)

    print(f"Saved: {path}")


# ============================================================
# MAIN
# ============================================================

def main():
    # --------------------------------------------------------
    # 1. Generate two reference matrices
    # --------------------------------------------------------

    D_ultra = generate_ultrametric_distance_matrix(
        n=N,
        seed=SEED,
        max_distance=MAX_DISTANCE,
    )

    D_additive = generate_additive_non_ultrametric_matrix(
        D_ultra,
        seed=SEED + 1000,
        pendant_scale=0.03,
        max_distance=MAX_DISTANCE,
    )

    matrices = {
        "synthetic_ultrametric": D_ultra,
        "synthetic_additive_nonultrametric": D_additive,
    }

    matrix_summary("Synthetic 30x30 exact ultrametric matrix", D_ultra)
    matrix_summary("Synthetic 30x30 additive non-ultrametric matrix", D_additive)

    # --------------------------------------------------------
    # 2. Save reference matrices
    # --------------------------------------------------------

    refs_dir = OUTDIR / "reference_matrices"
    refs_dir.mkdir(exist_ok=True)

    save_label_mapping(OUTDIR / "T_labels_mapping.csv")

    save_matrix(
        D_ultra,
        T_LABELS,
        refs_dir / "Dref_synthetic30_ultrametric_Tlabels.csv",
    )

    save_matrix(
        D_additive,
        T_LABELS,
        refs_dir / "Dref_synthetic30_additive_nonultrametric_Tlabels.csv",
    )

    save_matrix(
        D_ultra,
        FULL_LABELS,
        refs_dir / "Dref_synthetic30_ultrametric_full_labels.csv",
    )

    save_matrix(
        D_additive,
        FULL_LABELS,
        refs_dir / "Dref_synthetic30_additive_nonultrametric_full_labels.csv",
    )

    # Compatibility copy if old code expects this exact filename.
    # By default this points to the exact ultrametric matrix.
    save_matrix(
        D_ultra,
        T_LABELS,
        OUTDIR / "Dref_MAFFT_pairwise_deletion_pdistance_Tlabels.csv",
    )

    # --------------------------------------------------------
    # 3. Generate masks and observed matrices for BOTH matrices
    # --------------------------------------------------------

    masks_root = OUTDIR / "masks_and_observed"
    masks_root.mkdir(exist_ok=True)

    inventory = []
    n_pairs = N * (N - 1) // 2

    for matrix_type, D_ref in matrices.items():
        matrix_dir = masks_root / matrix_type
        matrix_dir.mkdir(exist_ok=True)

        for missing in MISSING_LEVELS:
            missing_tag = int(round(100 * missing))

            for rep in range(1, N_REPLICATES + 1):
                mask_seed = SEED + 10000 + missing_tag * 100 + rep

                # Same mask seed for both matrix types.
                # This makes ultrametric vs additive results directly comparable.
                mask = make_missing_mask(
                    n=N,
                    missing_fraction=missing,
                    seed=mask_seed,
                )

                D_obs = apply_missing_mask(D_ref, mask)

                n_missing = int(mask[np.triu_indices(N, k=1)].sum())
                n_observed = n_pairs - n_missing

                mask_int = mask.astype(int)

                mask_path = matrix_dir / (
                    f"mask_{matrix_type}_missing{missing_tag}_rep{rep:02d}.csv"
                )

                obs_path = matrix_dir / (
                    f"Dobs_{matrix_type}_missing{missing_tag}_rep{rep:02d}.csv"
                )

                save_matrix(mask_int, T_LABELS, mask_path)
                save_matrix(D_obs, T_LABELS, obs_path)

                if matrix_type == "synthetic_ultrametric":
                    ref_file = refs_dir / "Dref_synthetic30_ultrametric_Tlabels.csv"
                else:
                    ref_file = refs_dir / "Dref_synthetic30_additive_nonultrametric_Tlabels.csv"

                inventory.append({
                    "n": N,
                    "matrix_type": matrix_type,
                    "missing_fraction": missing,
                    "missing_percent": missing_tag,
                    "replicate": rep,
                    "n_pairs": n_pairs,
                    "n_missing": n_missing,
                    "n_observed": n_observed,
                    "reference_matrix_file": str(ref_file),
                    "mask_file": str(mask_path),
                    "observed_matrix_file": str(obs_path),
                    "seed": mask_seed,
                })

    inventory_df = pd.DataFrame(inventory)
    inventory_path = OUTDIR / "synthetic30_mask_inventory.csv"
    inventory_df.to_csv(inventory_path, index=False)

    print(f"\nSaved inventory: {inventory_path}")

    print("\nReference matrices:")
    print(f"  {refs_dir / 'Dref_synthetic30_ultrametric_Tlabels.csv'}")
    print(f"  {refs_dir / 'Dref_synthetic30_additive_nonultrametric_Tlabels.csv'}")

    print("\nCompatibility D_ref file for old code:")
    print(f"  {OUTDIR / 'Dref_MAFFT_pairwise_deletion_pdistance_Tlabels.csv'}")

    print("\nObserved matrices are saved in:")
    print(f"  {masks_root / 'synthetic_ultrametric'}")
    print(f"  {masks_root / 'synthetic_additive_nonultrametric'}")

    print("\nDone.")


if __name__ == "__main__":
    main()


Synthetic 30x30 exact ultrametric matrix
  shape:  (30, 30)
  min:    0.004369
  median: 0.224875
  mean:   0.199948
  max:    0.250000
  nan:    0
  max ultrametric violation: 0.000000000000e+00

Synthetic 30x30 additive non-ultrametric matrix
  shape:  (30, 30)
  min:    0.024344
  median: 0.162340
  mean:   0.156475
  max:    0.250000
  nan:    0
  max ultrametric violation: 6.515731478881e-02
Saved: synthetic30_tree_distances/T_labels_mapping.csv
Saved: synthetic30_tree_distances/reference_matrices/Dref_synthetic30_ultrametric_Tlabels.csv
Saved: synthetic30_tree_distances/reference_matrices/Dref_synthetic30_additive_nonultrametric_Tlabels.csv
Saved: synthetic30_tree_distances/reference_matrices/Dref_synthetic30_ultrametric_full_labels.csv
Saved: synthetic30_tree_distances/reference_matrices/Dref_synthetic30_additive_nonultrametric_full_labels.csv
Saved: synthetic30_tree_distances/Dref_MAFFT_pairwise_deletion_pdistance_Tlabels.csv
Saved: synthetic30_tree_distances/masks_and_observe